# EDA & PROFILING


## 1. Inisialisasi Pyspark Session


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Standar spark session lokal untuk profiling
spark = (
    SparkSession.builder
    .appName('Olist_EDA_Profiling')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

print(f'Spark initialized: {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 19:56:06 WARN Utils: Your hostname, lathief-laptop, resolves to a loopback address: 127.0.1.1; using 192.168.1.9 instead (on interface wlp0s20f3)
26/09/16 19:56:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lathief/coding/myproject/project4-aws/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/16 19:56:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark initialized: 4.2.0


## 2. Load Raw Bronze Data (Olist)


In [16]:
raw_path = '../data/raw/olist'

df_orders = spark.read.csv(f'{raw_path}/olist_orders_dataset.csv', header=True, inferSchema=True)
df_order_items = spark.read.csv(f'{raw_path}/olist_order_items_dataset.csv', header=True, inferSchema=True)
df_customers = spark.read.csv(f'{raw_path}/olist_customers_dataset.csv', header=True, inferSchema=True)
df_products = spark.read.csv(f'{raw_path}/olist_products_dataset.csv', header=True, inferSchema=True)
df_payments = spark.read.csv(f'{raw_path}/olist_order_payments_dataset.csv', header=True, inferSchema=True)
df_categories = spark.read.csv(f'{raw_path}/product_category_name_translation.csv', header=True, inferSchema=True)

print(f'Total Orders: {df_orders.count()}')
print(f'Total Order Items: {df_order_items.count()}')
print(f'Total Customers: {df_customers.count()}')
print(f'Total Products: {df_products.count()}')
print(f'Total Payments: {df_payments.count()}')
print(f'Total Categories: {df_categories.count()}')

Total Orders: 99441
Total Order Items: 112650
Total Customers: 99441
Total Products: 32951
Total Payments: 103886
Total Categories: 71


In [17]:
print('Schema Orders:')
df_orders.printSchema()
df_orders.select(
    F.col('*')
).show(5, truncate=False)

print('Schema Order Items:')
df_order_items.printSchema()
df_order_items.select(
    F.col('*')
).show(5, truncate=False)

print('Schema Customers:')
df_customers.printSchema()
df_customers.select(
    F.col('*')
).show(5, truncate=False)

print('Schema Products:')
df_products.printSchema()
df_products.select(
    F.col('*')
).show(5, truncate=False)

print('Schema Payments:')
df_payments.printSchema()
df_payments.select(
    F.col('*')
).show(5, truncate=False)

print('Schema Categories:')
df_categories.printSchema()
df_categories.select(
    F.col('*')
).show(5, truncate=False)

Schema Orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+---------------

## 3. Profiling & Grain Analysis (Fact Candidates)


In [20]:
# Cek order status
df_orders.groupBy('order_status').count().orderBy(F.desc('count')).show()

# Cek relasi orders dan order items
# satu order bisa memiliku banyak order items
df_order_items.groupBy('order_id').count().orderBy(F.desc('count')).show(5, truncate=False)

df_orders_ts = (
    df_orders
    .withColumn('purchase_ts', F.to_timestamp('order_purchase_timestamp'))
    .withColumn('purchase_date', F.to_date('purchase_ts'))
)

df_orders_ts.select(
    F.min('purchase_date').alias('min_purchase_date'),
    F.max('purchase_date').alias('max_purchase_date'),
    F.countDistinct('purchase_date').alias('total_days_active')
).show()



+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+

+--------------------------------+-----+
|order_id                        |count|
+--------------------------------+-----+
|8272b63d03f5f79c56e9e4120aec44ef|21   |
|1b15974a0141d54e36626dca3fdc731a|20   |
|ab14fdcfbe524636d65ee38360e22ce8|20   |
|428a2f660dc84138d969ccd69a0ab6d5|15   |
|9ef13efd6949e4573a18964dd1bbe7f5|15   |
+--------------------------------+-----+
only showing top 5 rows
+-----------------+-----------------+-----------------+
|min_purchase_date|max_purchase_date|total_days_active|
+-----------------+-----------------+-----------------+
|       2016-09-04|       2018-10-17|              634|
+-----------------+-----------------+-----------------+



## 4. Analisis Dimensi Customer


In [27]:
# Periksa perbedaan customer_id dengan customer_unique_id

total_cust_rows = df_customers.count()
distinct_cust_id = df_customers.select('customer_unique_id').distinct().count()

print(f'Total customer records (per transaksi): {total_cust_rows}')
print(f'Total unique individuals: {distinct_cust_id}')
print(f'Repeat customers: {total_cust_rows - distinct_cust_id}')

repeat_customers = (
    df_customers
    .groupBy('customer_unique_id')
    .agg(
        F.count('customer_id').alias('order_count'),
        F.countDistinct('customer_city').alias('city_count'),
        F.countDistinct('customer_state').alias('state_count')
    )
    .filter(
        F.col('city_count') > 1
    )
    .orderBy(F.desc('city_count'))
)

print('Customer yang berpindah kota/lokasi:')
repeat_customers.show(5, truncate=False)

Total customer records (per transaksi): 99441
Total unique individuals: 96096
Repeat customers: 3345
Customer yang berpindah kota/lokasi:
+--------------------------------+-----------+----------+-----------+
|customer_unique_id              |order_count|city_count|state_count|
+--------------------------------+-----------+----------+-----------+
|d44ccec15f5f86d14d6a2cfa67da1975|3          |3         |3          |
|5275b2f97b9c995d3d05a58610c0bb67|2          |2         |2          |
|9935b7e2683de890a208026f020e54d3|2          |2         |1          |
|ce41a50f9ff95ef577cb759483fe9165|2          |2         |1          |
|68f412bbb674141ab49ef13b501baf45|2          |2         |1          |
+--------------------------------+-----------+----------+-----------+
only showing top 5 rows


## 5. Analisis Dimensi Produk


In [31]:
# Cek missing values dan kategori

df_products.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_products.columns
    ]
).show(truncate=False)

df_products_enriched = df_products.join(
    df_categories,
    on='product_category_name',
    how='left'
)

df_products_enriched.select(
    'product_id',
    'product_category_name',
    'product_category_name_english'
).show(5, truncate=False)

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0         |610                  |610                |610                       |610               |2               |2                |2                |2               |
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+

+--------------------------------+---------------------+-----------------------------+
|product_id                      |product_category_name|p